# មេរៀន ០២ - ស្វែងរក Microsoft Agent Framework

**Microsoft Agent Framework (MAF)** គឺជាគ្រោងបណ្ដាញតែមួយសម្រាប់បង្កើតភ្នាក់ងារជំនួយប្រាជ្ញាសិប្បនិម្មិត។ វាបង្ហាញពីសំណង់ស្អាតនិងអាចបញ្ចូលគ្នាបានជាមួយប្លុកស្ថាបនាចម្បងបួន:

- **Client** – ភ្ជាប់ទៅនឹងចំណុចបញ្ចប់ម៉ូដែល AI និងគ្រប់គ្រងការប្រាស្រ័យទាក់ទង
- **Agent** – ប្រអប់ client ជាមួយការណែនាំនិងការកំណត់ឧបករណ៍
- **Tools** – ពង្រីកសមត្ថភាពភ្នាក់ងារជាមួយមុខងារផ្ទាល់ខ្លួនដែលម៉ូដែលអាចហៅបាន
- **Session** – រក្សាទុកប្រវត្តិការនិយាយសម្រាប់ការផ្លាស់ប្ដូរក្នុងចំណោមច្រើនជុំវិញ

នៅក្នុងមេរៀននេះ យើងនឹងបង្កើត **ភ្នាក់ងារកក់ដំណើរកម្សាន្ត** ដែលពិនិត្យភាពអាចប្រើបាននៃគោលដៅដោយប្រើគំនិតទាំងនេះ។


## ការតំឡើង


In [ ]:
# Install the Microsoft Agent Framework package
! pip install agent-framework azure-ai-projects -U -q
! pip install python-dotenv -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import dotenv
from typing import Annotated

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential

dotenv.load_dotenv(dotenv.find_dotenv())

## ការយល់ដឹងអំពីស្ថាបត្យកម្ម Agent Framework

Microsoft Agent Framework បានអនុវត្តស្ថាបត្យកម្មជាស្រទាប់៖

```
Client  →  Agent  →  Tools
                  →  Session
```

1. **Client** – អ្នកប្រើប្រាស់ `FoundryChatClient` ភ្ជាប់ទៅកាន់ការប្រើប្រាស់ Azure OpenAI។ វាដំណើរការការផ្ទៀងផ្ទាត់, ការរៀបចំសំណើ, និងការបកស្រាយចម្លើយ។
2. **Agent** – បង្កើតពី client តាមរយៈ `provider.create_agent()`, agent ផ្គូរផ្គងការចូលប្រើម៉ូដែលជាមួយនឹងការណែនាំ (system prompt) និងឧបករណ៍។
3. **Tools** – មុខងារ Python ដែលបានតុបតែងជាមួយ `@tool` ដែល agent អាចហៅដើម្បីអនុវត្តសកម្មភាពឬយកទិន្នន័យ។
4. **Session** – វត្ថុ `AgentSession` (បង្កើតដោយ `agent.create_session()`) ដែលរក្សាប្រវត្តិសន្ទនា, ធ្វើឲ្យអាចមានការសន្ទនាច្រើនជំហានដែល agent ចងចាំបរិបទមុនគេ។

យើងទៅបង្កើតនីមួយៗតាមជាន់ជំហាន។


In [ ]:
# Create the client – this is the connection to the AI model
endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
model = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

if not endpoint or not model:
    raise ValueError(
        "Missing required environment variables. "
        "Please set AZURE_AI_PROJECT_ENDPOINT and AZURE_AI_MODEL_DEPLOYMENT_NAME as environment variables (e.g., in your .env file or shell environment)."
    )

provider = FoundryChatClient(
    project_endpoint=endpoint,
    model=model,
    credential=AzureCliCredential()
)

## ការបន្ថែមឧបករណ៍ជាមួយ @tool Decorator

ឧបករណ៍អនុញ្ញាតឱ្យភ្នាក់ងារធ្វើសកម្មភាពក្រៅពីការបង្កើតអត្ថបទ។ @tool decorator បម្លែងមុខងារ Python ធម្មតាទៅជារបស់ដែលភ្នាក់ងារអាចហៅបាន។

ចំណុចសំខាន់ៗ៖
- ប្រើ `Annotated[type, "description"]` ដើម្បីឱ្យម៉ូដែលយល់ពីប៉ារ៉ាម៉ែត្រតិចតួច។
- docstring ក្លាយទៅជាការពិពណ៌នាឧបករណ៍ដែលម៉ូដែលឃើញ។
- `approval_mode="never_require"` មានន័យថាឧបករណ៍ដំណើរការត автоматичноដោយគ្មានការបញ្ជាក់ពីអ្នកប្រើ។


In [ ]:
@tool(approval_mode="never_require")
def check_destination_availability(
    destination: Annotated[str, "The destination to check availability for"]
) -> str:
    """Check if a vacation destination is currently available for booking."""
    available = {
        "Barcelona": True,
        "Tokyo": True,
        "Cape Town": False,
        "Vancouver": True,
        "Dubai": False,
    }
    is_available = available.get(destination, False)
    return f"{destination} is {'available' if is_available else 'not available'} for booking."

## ការបង្កើតភ្នាក់ងារជាមួយឧបករណ៍

ឥឡូវនេះយើងផ្សំគ្នារវាងអតិថិជន, ការណែនាំ, និងឧបករណ៍ទៅជាភ្នាក់ងារ។ `instructions` ធ្វើដូចជាប្រមុខប្រព័ន្ធ — ពួកវាបញ្ជាក់អត្តសញ្ញាណ និងអាកប្បកិរិយារបស់ភ្នាក់ងារ។


In [ ]:
agent = provider.as_agent(
    name="TravelAvailabilityAgent",
    instructions=(
        "You are a travel booking agent. Help users check destination availability "
        "and make recommendations. Always check availability before recommending a destination."
    ),
    tools=[check_destination_availability],
)

## ការសន្ទនារយៈពេលច្រើនជុំជាមួយកម្មវិធីសម័យ

`AgentSession` មួយ(បង្កើតតាមរយៈ `agent.create_session()`) រក្សាទុកសារ​ទាំងអស់​ក្នុង​ការសន្ទនា។ ដោយផ្តល់សម័យដូចគ្នាទៅកាន់ការហៅ `agent.run()` រៀងរាល់ពេល​ អ្នកប្រើប្រាស់អាចចូលដំណើរការ​ទិសដៅសន្ទនាប្រវត្តិសាស្ត្រីទាំងមូល ហើយអាចយោងទៅកាន់សារដែលមានមុន។

យើងផ្តល់ `tools=[check_destination_availability]` ដូច្នេះអ្នកប្រើប្រាស់អាចហៅកម្មវិធីត្រួតពិនិត្យភាពមានស្រាប់របស់វា នៅពេលរៀងរាល់ជុំ។ 


In [ ]:
session = agent.create_session()

# Turn 1: Ask about available destinations
response = await agent.run(
    "Which destinations do you have available?",
    session=session,
)
print(f"Agent: {response}")

In [ ]:
# Turn 2: Follow-up question — the agent remembers the conversation
response = await agent.run(
    "I'd like to go somewhere warm. What's available?",
    session=session,
)
print(f"Agent: {response}")

## សេចក្ដីសង្ខេប

ក្នុងមេរៀននេះ អ្នកបានស្វែងយល់ពីស្ថম্ভសំខាន់បួននៃ Microsoft Agent Framework៖

| គំនិត | អ្វីដែលអ្នកបានរៀន |
|---------|------------------|
| **Client** | `FoundryChatClient` អាចភ្ជាប់ទៅ Azure OpenAI ជាមួយការផ្ទៀងផ្ទាត់ដោយគ្រឿងសញ្ញា |
| **Agent** | `provider.create_agent()` បញ្ជាក់ការតភ្ជាប់ម៉ូដែលជាមួយបញ្ជីណែនាំ និងឈ្មោះ |
| **Tools** | `@tool` decorator បង្ហាញមុខងារពី Python ដើម្បីឲ្យ agent អាចហៅបាន |
| **Session** | `agent.create_session()` រក្សាប្រវត្តិការជជែកអំឡុងរយៈពេលជាច្រើនជុំ |

គ្រឿងផ្សំទាំងនេះផ្សំឡើងគ្នាដើម្បីបង្កើតមុខងារ agent ដែលអាចអភិបាលសន្ទនាធម្មជាតិកាន់តែប្រសើរ ហៅមុខងារផ្សេងៗខាងក្រៅ ហើយរក្សាបរិបទ — ជាមូលដ្ឋានសម្រាប់លំនាំ agentic លម្អិតជាងនេះនៅមេរៀនក្រោយៗ។ 


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ការបដិសេធ**:
ឯកសារនេះត្រូវបានបម្លែងភាសា ដោយប្រើសេវាបម្លែងភាសា AI [Co-op Translator](https://github.com/Azure/co-op-translator)។ ទោះយើងខ្ញុំមានក្តីប្រាថ្នាឱ្យបានច្បាស់លាស់ តែសូមយល់ដឹងថាការបម្លែងដោយស្វ័យប្រវត្តិក៏អាចមានកំហុសឬភាពមិនត្រឹមត្រូវ។ ឯកសារដើមជាភាសាទីតាំងគួរត្រូវបានគេប្រើជាប្រភពច្បាស់លាស់។ សម្រាប់ព័ត៌មានសំខាន់ៗ សូមណែនាំឱ្យប្រើប្រាស់ការប្រែដោយមនុស្សជំនាញ។ យើងខ្ញុំមិនទទួលខុសត្រូវចំពោះការយល់ច្រឡំ ឬការបកស្រាយខុសបន្ទាប់ពីការប្រើប្រាស់ការបម្លែងនេះនោះទេ។
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
